Process for creating an experiment:

Things we input:
- How many steps each simulation should have.
- Perhaps a trader in question - could start by just graphing their wealth, or checking their wealth at the end.
    - if its specifically for one trader and we want the results to be average wealth accumulated or rate of wealth gain - wpould just be wealth at end - start / time steps or could do sharpe ratio
    - we can then define the trader and their strategies

- 

ideally we want to change to something where we can add in the starting levels, maybe we do an apply level function copying the live data version.

In [19]:
from dataclasses import dataclass

from traders import Trader

@dataclass
class experiment:
    target: Trader
    independent_variable: list

    other_traders: list[Trader]

    num_steps: int
    seeds: list[int]
    

we have our target trader.

1. fix first independent variable value from list, 
2. then run sim on each seed for num_steps. 
3. Record targets - ie append to a list things like avg pnl per step, sharpe ratio, max drawdown if thats appropriate.
4. run over each seed then average the values.
5. move to next value for independent variable.
6. produce graphs of the average for each of the items in question!

In [20]:
'''Factory - function? should be able to create a simulation or populate it with traders that have strategies.
'''
import sys
sys.path.insert(0, "..")

from simulation import Simulation
from strategies import Strategy, random_strat, inventorymm, naivemm

from traders import Trader
import json
import random
from dataclasses import dataclass


'''config = {
    "random_trader" : {
            qty: 80,
            trader_cfg: {},
            strategy_cfg: {},
        },
    "naive_market_maker": {
            qty: 20,
            trader_cfg: {},
            strategy_cfg: {},
        },
    }'''


@dataclass
class TraderConfig:
    min_starting_position: int = 100
    max_starting_position: int = 100
    min_starting_cash: int = 10000
    max_starting_cash: int = 10000
    min_latency: int = 1
    max_latency: int = 100
    min_latency_dev: int = 0
    max_latency_dev: int = 20

@dataclass
class RandomConfig:
    min_buy_prob: float
    max_buy_prob: float
    min_sell_prob: float
    max_sell_prob: float
    min_cancel_prob: float
    max_cancel_prob: float
    min_limit_prob: float
    max_limit_prob: float
    min_max_quantity: int
    max_max_quantity: int
    min_max_price_offest: int
    max_max_price_offset: int
    reference_price: int

@dataclass
class NaiveMMConfig:
    min_half_spread: int
    max_half_spread: int
    min_time_req: int
    max_time_req: int
    min_tolerance: int
    max_tolerance: int
    min_quantity: int
    max_quantity: int


@dataclass
class InventoryMMConfig:
    min_half_spread: int
    max_half_spread: int
    min_time_req: int
    max_time_req: int
    min_tolerance: int
    max_tolerance: int
    min_quantity: int
    max_quantity: int
    min_target_ratio: float
    max_target_ratio: float
    min_ratio_error: float
    max_ratio_error: float


class ConfigParser:
    def __init__(self):
        self.map = {
            "random_trader": self.parse_random_trader,
            "naive_mm": self.parse_naive_mm,
            "inventory_aware_mm": self.parse_inventory_mm,
        }

    def parse_base_trader(self, config: dict) -> TraderConfig:
        try:
            return TraderConfig(**config)
        except TypeError as e:
            raise ValueError(f"Invalid trader configuration: {e}") from e
        

    def parse_random_trader(self, config: dict) -> RandomConfig:
        try:
            return RandomConfig(**config)
        except TypeError as e:
            raise ValueError(f"Invalid strategy configuration: {e}") from e
        

    def parse_naive_mm(self, config: dict):
        try:
            return NaiveMMConfig(**config)
        except TypeError as e:
            raise ValueError(f"Invalid strategy configuration: {e}") from e

        
    def parse_inventory_mm(self, config: dict):
        try:
            return InventoryMMConfig(**config)
        except TypeError as e:
            raise ValueError(f"Invalid strategy configuration: {e}") from e


class TraderFactory:
    def __init__(self, seed):
        self.rng = random.Random(seed)
        self.map = {
            "random_trader": self.create_random,
            "naive_mm": self.create_naive_mm,
            "inventory_aware_mm": self.create_inventory_mm,
        }

    def create_trader(self, data: tuple[str, TraderConfig, RandomConfig | NaiveMMConfig | InventoryMMConfig], id: int, sim: Simulation) -> Trader:
        # unpack data
        strat = self.map[data[0]]
        trader_cfg = data[1]
        strat_cfg = data[2]

        return Trader(model=sim,
                      trader_id=id,
                      initial_cash=self.rng.randint(trader_cfg.min_starting_cash, trader_cfg.max_starting_cash),
                      initial_position=self.rng.randint(trader_cfg.min_starting_position, trader_cfg.max_starting_position),
                      latency=self.rng.randint(trader_cfg.min_latency, trader_cfg.max_latency),
                      latency_deviation=self.rng.randint(trader_cfg.min_latency_dev, trader_cfg.max_latency_dev),
                      strategy=strat(strat_cfg),
                      )


    def create_random(self, data: RandomConfig) -> Strategy:
        min_total = data.min_buy_prob + data.min_sell_prob
        both_fixed = (data.min_buy_prob == data.max_buy_prob 
                      and data.min_sell_prob == data.max_sell_prob)
        

        if min_total > 1 or (min_total == 1 and not both_fixed):
            raise ValueError(
                "Buy/sell ranges must allow sampling probabilities "
                "whose sum is at most 1."
            )

        buy_prob = self.rng.uniform(data.min_buy_prob, data.max_buy_prob)
        sell_prob = self.rng.uniform(data.min_sell_prob, data.max_sell_prob)

        while buy_prob + sell_prob > 1:
            buy_prob = self.rng.uniform(data.min_buy_prob, data.max_buy_prob)
            sell_prob = self.rng.uniform(data.min_sell_prob, data.max_sell_prob)

        return random_strat.Random(
            buy_probability=buy_prob,
            sell_probability=sell_prob,
            cancel_probability=self.rng.uniform(data.min_cancel_prob, data.max_cancel_prob),
            limit_probability=self.rng.uniform(data.min_limit_prob, data.max_limit_prob),
            max_quantity=self.rng.randint(data.min_max_quantity, data.max_max_quantity),
            max_price_offset=self.rng.randint(data.min_max_price_offest, data.max_max_price_offset),
            reference_price=data.reference_price,
        )



    def create_naive_mm(self, data: NaiveMMConfig) -> Strategy:
        return naivemm.NaiveMM(
            half_spread=self.rng.randint(data.min_half_spread, data.max_half_spread),
            time_req=self.rng.randint(data.min_time_req, data.max_time_req),
            tolerance=self.rng.randint(data.min_tolerance, data.max_tolerance),
            quantity=self.rng.randint(data.min_quantity, data.max_quantity),
        )

    def create_inventory_mm(self, data: InventoryMMConfig) -> Strategy:
        return inventorymm.InventoryMM(
            half_spread=self.rng.randint(data.min_half_spread, data.max_half_spread),
            time_req=self.rng.randint(data.min_time_req, data.max_time_req),
            tolerance=self.rng.randint(data.min_tolerance, data.max_tolerance),
            target_ratio=self.rng.uniform(data.min_target_ratio, data.max_target_ratio),
            ratio_error=self.rng.uniform(data.min_ratio_error, data.max_ratio_error),
            quantity=self.rng.randint(data.min_quantity, data.max_quantity),
        )

    



def factory(trader_config: str, decision_seed, trader_parameter_seed) -> Simulation:
    sim: Simulation = Simulation(seed=decision_seed)
    parser: ConfigParser = ConfigParser()
    trader_factory: TraderFactory = TraderFactory(seed=trader_parameter_seed)

    trader_id = 1


    config = json.loads(trader_config)

    # create the config classes, get their quantities:
    config_classes = []


    for strategy in config:
        try:
            parse_function = parser.map[strategy]
        except KeyError:
            raise ValueError(f"Unknown trader strategy: {strategy}")

        qty = config[strategy]["quantity"]
        trader_cfg = parser.parse_base_trader(config[strategy]["trader_cfg"])
        strategy_cfg = parse_function(config[strategy]["strategy_cfg"])

        config_classes.append(((strategy, trader_cfg, strategy_cfg), qty))

    # config_classes is now a list of tuples: [(strategy_name, quantity, trader_config, strategy_config)]
    # want to process each 

    for config, qty in config_classes:
        for i in range(qty):
            # pass in config to create trader:
            trader: Trader = trader_factory.create_trader(config, trader_id, sim=sim)
            sim.traders[trader_id] = trader
            trader_id += 1


    return sim







In [21]:
trader_config = '''{
  "random_trader": {
    "quantity": 20,
    "trader_cfg": {},
    "strategy_cfg": {
      "min_buy_prob": 0.2,
      "max_buy_prob": 0.6,
      "min_sell_prob": 0.2,
      "max_sell_prob": 0.6,
      "min_cancel_prob": 0.01,
      "max_cancel_prob": 0.1,
      "min_limit_prob": 0.5,
      "max_limit_prob": 1.0,
      "min_max_quantity": 1,
      "max_max_quantity": 5,
      "min_max_price_offest": 0,
      "max_max_price_offset": 3,
      "reference_price": 100
    }
  },
  "naive_mm": {
    "quantity": 5,
    "trader_cfg": {},
    "strategy_cfg": {
      "min_half_spread": 1,
      "max_half_spread": 2,
      "min_time_req": 1,
      "max_time_req": 5,
      "min_tolerance": 0,
      "max_tolerance": 2,
      "min_quantity": 1,
      "max_quantity": 5
    }
  },
  "inventory_aware_mm": {
    "quantity": 5,
    "trader_cfg": {},
    "strategy_cfg": {
      "min_half_spread": 1,
      "max_half_spread": 2,
      "min_time_req": 1,
      "max_time_req": 5,
      "min_tolerance": 0,
      "max_tolerance": 2,
      "min_quantity": 1,
      "max_quantity": 5,
      "min_target_ratio": 0.4,
      "max_target_ratio": 0.6,
      "min_ratio_error": 0.05,
      "max_ratio_error": 0.1
    }
  }
}'''

In [22]:
sim = factory(trader_config=trader_config,
              decision_seed=42,
              trader_parameter_seed=123)

In [32]:
sim.run_sim(10000)

In [33]:
for trader_id, trader in sim.traders.items():
    print(
        f"Trader {trader_id}: "
        f"strategy={type(trader.strategy).__name__}, "
        f"cash={trader.current_cash}, "
        f"position={trader.current_position}"
    )

Trader 1: strategy=Random, cash=11962.5, position=76.0
Trader 2: strategy=Random, cash=6911.75, position=129.0
Trader 3: strategy=Random, cash=9085.75, position=116.0
Trader 4: strategy=Random, cash=11522.5, position=84.0
Trader 5: strategy=Random, cash=10506.25, position=94.0
Trader 6: strategy=Random, cash=16498.25, position=36.0
Trader 7: strategy=Random, cash=6769.25, position=137.0
Trader 8: strategy=Random, cash=13071.5, position=69.0
Trader 9: strategy=Random, cash=8745.75, position=113.0
Trader 10: strategy=Random, cash=13746.0, position=63.0
Trader 11: strategy=Random, cash=6382.75, position=137.0
Trader 12: strategy=Random, cash=12003.5, position=80.0
Trader 13: strategy=Random, cash=9937.0, position=100.0
Trader 14: strategy=Random, cash=13005.5, position=70.0
Trader 15: strategy=Random, cash=9976.75, position=100.0
Trader 16: strategy=Random, cash=9168.5, position=108.0
Trader 17: strategy=Random, cash=5746.75, position=141.0
Trader 18: strategy=Random, cash=7925.5, positio